# Pangenome analysis with PPanGGOLiN
Opened from **BNERC**, the Bionitrogen Fixation Research Database (BicBioEng lab, University of South Dakota).

Downloads each selected genome, then annotates, clusters and partitions them into core, shell and cloud genes with PPanGGOLiN, and draws the U-curve and tile plot. Use at least 3 genomes.

**How to run:** paste the records BNERC copied into `RECORDS` below, then choose **Runtime → Run all**. Installing PPanGGOLiN takes a few minutes.

In [ ]:
#@title Records to analyze
#@markdown Paste what BNERC copied for you: `accession:source` pairs separated by commas. The source can be left out; it's inferred from the accession.
RECORDS = "GCA_000009705:cyanobacteria_ena, GCA_000020025:cyanobacteria_ena"  #@param {type:"string"}
#@markdown NCBI asks for a contact email with each download request.
NCBI_EMAIL = "readstech@gmail.com"  #@param {type:"string"}

def _parse(text):
    out = []
    for part in text.replace("\n", ",").split(","):
        part = part.strip()
        if not part:
            continue
        acc, _, src = part.partition(":")
        acc, src = acc.strip(), src.strip()
        if not src:
            src = "cyanobacteria_ncbi" if acc.upper().startswith(("GCA_", "GCF_")) else "cyanorak"
        out.append((acc, src))
    return out

records = _parse(RECORDS)
accessions = [a for a, _ in records]
assert records, "Paste at least one record into RECORDS above."
print(f"{len(records)} record(s):", ", ".join(f"{a} ({s})" for a, s in records))

## 1. Install PPanGGOLiN (via Miniconda)

In [ ]:
!wget -nc -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!bash Miniconda3-latest-Linux-x86_64.sh -b -f -p /usr/local > /dev/null
!conda install -y -q -c conda-forge -c bioconda ppanggolin > /dev/null
!pip -q install biopython
!ppanggolin --version

## 2. Download sequences

In [ ]:
#@title Download genomes as FASTA (.fna)
import glob, os, shutil, subprocess, urllib.request, zipfile
from Bio import Entrez
Entrez.email = NCBI_EMAIL

def _datasets_cli():
    exe = shutil.which("datasets") or os.path.abspath("bin/datasets")
    if not os.path.exists(exe):
        os.makedirs("bin", exist_ok=True)
        urllib.request.urlretrieve("https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/LATEST/linux-amd64/datasets", exe)
        os.chmod(exe, 0o755)
    return exe

def fetch(acc, source):
    dest = f"{acc}.fna"
    if os.path.exists(dest):
        return print(f"✅ {dest} already downloaded")
    try:
        if source in ("cyanobacteria_ncbi", "cyanobacteria_gtdb"):
            # Assembly accessions (GCA_/GCF_): NCBI Datasets CLI
            subprocess.run([_datasets_cli(), "download", "genome", "accession", acc, "--include", "genome", "--filename", f"{acc}.zip"], check=True)
            with zipfile.ZipFile(f"{acc}.zip") as z:
                z.extractall(f"genome_{acc}")
            hits = glob.glob(f"genome_{acc}/ncbi_dataset/data/*/*.fna")
            if not hits:
                return print(f"❌ No .fna in the NCBI package for {acc}")
            shutil.move(hits[0], dest)
        elif source == "cyanobacteria_ena":
            urllib.request.urlretrieve(f"https://www.ebi.ac.uk/ena/browser/api/fasta/{acc}?download=true", dest)
        else:
            # cyanorak / cyanobacteria_ncbi_nucelotide: NCBI Nucleotide via Entrez
            with Entrez.efetch(db="nucleotide", id=acc, rettype="fasta", retmode="text") as h:
                open(dest, "w").write(h.read())
        print(f"✅ {dest} ({os.path.getsize(dest) // 1024} KB)")
    except Exception as e:
        print(f"❌ {acc} ({source}): {e}")

for acc, src in records:
    fetch(acc, src)
missing = [a for a in accessions if not os.path.exists(f"{a}.fna") or os.path.getsize(f"{a}.fna") == 0]
if missing:
    print("⚠️ Not downloaded:", ", ".join(missing))

## 3. Genome list for PPanGGOLiN

In [ ]:
os.makedirs("genomes_fna", exist_ok=True)
lines = []
for acc in accessions:
    if os.path.exists(f"{acc}.fna"):
        shutil.copy(f"{acc}.fna", f"genomes_fna/{acc}.fna")
        lines.append(f"{acc}\t{os.path.abspath(f'genomes_fna/{acc}.fna')}")
open("genomes_list.tsv", "w").write("\n".join(lines) + "\n")
print(len(lines), "genomes listed")
assert len(lines) >= 2, "Pangenome analysis needs at least 2 downloaded genomes (3+ recommended)."

## 4. Annotate, cluster, build the graph, partition

In [ ]:
!ppanggolin annotate --fasta genomes_list.tsv --output annotated --force
!ppanggolin cluster -p annotated/pangenome.h5 --identity 0.5 --coverage 0.8 --force
!ppanggolin graph -p annotated/pangenome.h5 --force
!ppanggolin partition -p annotated/pangenome.h5 --force
!ppanggolin info -p annotated/pangenome.h5 --content

## 5. Export and plots

In [ ]:
os.makedirs("output", exist_ok=True)
!ppanggolin write_pangenome -p annotated/pangenome.h5 --csv --output output/presence_absence --force
!ppanggolin write_pangenome -p annotated/pangenome.h5 --light_gexf --output output/pangenome_graph --force
!ppanggolin draw -p annotated/pangenome.h5 --ucurve --output output/fig_ucurve --force
!ppanggolin draw -p annotated/pangenome.h5 --tile_plot --nocloud --output output/fig_tile --force
from IPython.display import HTML, display
for f in glob.glob("output/fig_*/*.html"):
    print(f)
    display(HTML(open(f).read()))